# MNIST 手写数字识别 - Transformer 演示

本 notebook 展示如何使用 Transformer 编码器对 MNIST 手写数字进行分类。
模型将 28x28 的图像视为 784 个 token 的序列进行处理。

**完全自包含，无需外部模块导入。**

## 1. 环境设置

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import ipywidgets as widgets
from PIL import Image
import io

# 设置设备
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

## 2. 配置参数

In [ ]:
# 数据常量
MNIST_MEAN = 0.1307
MNIST_STD = 0.3081

# 训练配置
config = {
    'data_dir': './data',
    'batch_size': 64,
    'epochs': 10,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'val_split': 0.1,
    
    # 模型配置
    'd_model': 64,
    'nhead': 4,
    'num_layers': 4,
    'dim_feedforward': 128,
    'dropout': 0.1,
    'num_classes': 10,
    
    # 训练控制
    'grad_clip_norm': 1.0,
    'early_stopping_patience': 5,
}

print('Configuration:')
for k, v in config.items():
    print(f'  {k}: {v}')

## 3. 模型定义

In [ ]:
class SequenceTransformerClassifier(nn.Module):
    """将 28x28 图像序列化为 784 个 token，使用 Transformer 编码器分类。"""

    def __init__(
        self,
        d_model: int = 64,
        nhead: int = 4,
        num_layers: int = 4,
        dim_feedforward: int = 128,
        dropout: float = 0.1,
        num_classes: int = 10,
    ) -> None:
        super().__init__()
        self.seq_len = 28 * 28

        self.token_proj = nn.Linear(1, d_model)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.seq_len, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(
            encoder_layer=encoder_layer, num_layers=num_layers, enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

        self._init_weights()

    def _init_weights(self) -> None:
        nn.init.xavier_uniform_(self.token_proj.weight)
        nn.init.zeros_(self.token_proj.bias)
        nn.init.normal_(self.pos_embed, std=0.02)
        nn.init.xavier_uniform_(self.head.weight)
        nn.init.zeros_(self.head.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, 1, 28, 28)
        bsz = x.shape[0]
        x = x.view(bsz, self.seq_len, 1)
        x = self.token_proj(x)
        x = x + self.pos_embed
        x = self.encoder(x)
        x = self.norm(x)
        x = x.mean(dim=1)
        logits = self.head(x)
        return logits

In [ ]:
# 创建模型
model = SequenceTransformerClassifier(
    d_model=config['d_model'],
    nhead=config['nhead'],
    num_layers=config['num_layers'],
    dim_feedforward=config['dim_feedforward'],
    dropout=config['dropout'],
    num_classes=config['num_classes'],
).to(device)

print(model)
print(f'\n模型参数量: {sum(p.numel() for p in model.parameters()):,}')

## 4. 数据加载

In [ ]:
def get_dataloaders(data_dir, batch_size, val_split=0.1, seed=42):
    """获取 MNIST 数据加载器"""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
    ])

    full_train = datasets.MNIST(root=data_dir, train=True, download=True, transform=transform)
    test_set = datasets.MNIST(root=data_dir, train=False, download=True, transform=transform)

    val_size = int(len(full_train) * val_split)
    train_size = len(full_train) - val_size

    generator = torch.Generator().manual_seed(seed)
    train_set, val_set = random_split(full_train, [train_size, val_size], generator=generator)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

# 加载数据
train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=config['data_dir'],
    batch_size=config['batch_size'],
    val_split=config['val_split'],
)

print(f'训练集批次数: {len(train_loader)}')
print(f'验证集批次数: {len(val_loader)}')
print(f'测试集批次数: {len(test_loader)}')

In [ ]:
# 可视化样本
def show_samples(loader, num_samples=16):
    images, labels = next(iter(loader))
    fig, axes = plt.subplots(4, 4, figsize=(8, 8))
    for i, ax in enumerate(axes.flat):
        if i < num_samples:
            img = images[i].squeeze() * MNIST_STD + MNIST_MEAN
            ax.imshow(img, cmap='gray')
            ax.set_title(f'Label: {labels[i].item()}')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(train_loader)

## 5. 训练

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, grad_clip_norm=1.0):
    """训练一个 epoch"""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip_norm)
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    """评估模型"""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    return total_loss / len(loader), correct / total

In [ ]:
# 训练设置
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['epochs'])

# 记录训练历史
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

# Early stopping
best_val_acc = 0.0
patience_counter = 0
best_model_state = None

In [ ]:
# 训练循环
print('开始训练...')
for epoch in range(config['epochs']):
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, config['grad_clip_norm']
    )
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f'Epoch {epoch+1}/{config["epochs"]}: '
          f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
    
    # Early stopping
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        best_model_state = model.state_dict().copy()
    else:
        patience_counter += 1
        if patience_counter >= config['early_stopping_patience']:
            print(f'Early stopping at epoch {epoch+1}')
            break

# 加载最佳模型
if best_model_state is not None:
    model.load_state_dict(best_model_state)
    print(f'\n加载最佳模型 (Val Acc: {best_val_acc:.4f})')

## 6. 训练曲线可视化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss 曲线
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy 曲线
axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training and Validation Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# 测试集评估
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f'测试集 Loss: {test_loss:.4f}, Accuracy: {test_acc:.4f}')

## 7. 交互式演示 - 手写数字识别

在下方画布上绘制数字 (0-9)，然后点击预测按钮查看结果。

In [ ]:
# 创建画布组件
canvas_width = 280
canvas_height = 280

# 使用 HTML Canvas 进行绘图
canvas_html = """
<canvas id='draw-canvas' width='{width}' height='{height}' 
        style='border:2px solid #333; background:black; cursor:crosshair;'></canvas>
""".format(width=canvas_width, height=canvas_height)

# JavaScript 处理绘图
js_code = """
<script>
var canvas = document.getElementById('draw-canvas');
var ctx = canvas.getContext('2d');
var drawing = false;
var lastX = 0, lastY = 0;

ctx.strokeStyle = 'white';
ctx.lineWidth = 20;
ctx.lineCap = 'round';
ctx.lineJoin = 'round';

canvas.addEventListener('mousedown', function(e) {
    drawing = true;
    lastX = e.offsetX;
    lastY = e.offsetY;
});

canvas.addEventListener('mousemove', function(e) {
    if (drawing) {
        ctx.beginPath();
        ctx.moveTo(lastX, lastY);
        ctx.lineTo(e.offsetX, e.offsetY);
        ctx.stroke();
        lastX = e.offsetX;
        lastY = e.offsetY;
    }
});

canvas.addEventListener('mouseup', function() { drawing = false; });
canvas.addEventListener('mouseout', function() { drawing = false; });

window.clearCanvas = function() {
    ctx.fillStyle = 'black';
    ctx.fillRect(0, 0, canvas.width, canvas.height);
};

window.getCanvasData = function() {
    return canvas.toDataURL('image/png');
};

// 初始化画布
clearCanvas();
</script>
"""

display(widgets.HTML(canvas_html + js_code))

# 按钮
clear_btn = widgets.Button(description='清除画布')
predict_btn = widgets.Button(description='预测', button_style='success')
output = widgets.Output()

def clear_canvas(b):
    display(widgets.HTML("<script>clearCanvas();</script>"))
    with output:
        clear_output()

def predict(b):
    with output:
        clear_output()
        print('请在画布上绘制数字，然后点击预测按钮。')
        print('(注意: 此演示需要在支持 JavaScript 的环境中运行)')

clear_btn.on_click(clear_canvas)
predict_btn.on_click(predict)

display(widgets.HBox([clear_btn, predict_btn]))
display(output)

In [ ]:
# 简化版演示: 使用测试集样本进行预测展示
print('使用测试集样本进行预测演示:')
print('-' * 50)

# 获取一些测试样本
test_images, test_labels = next(iter(test_loader))
test_images = test_images[:8].to(device)
test_labels = test_labels[:8]

# 预测
model.eval()
with torch.no_grad():
    outputs = model(test_images)
    probs = torch.softmax(outputs, dim=1)
    preds = outputs.argmax(dim=1)

# 可视化
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    img = test_images[i].cpu().squeeze() * MNIST_STD + MNIST_MEAN
    ax.imshow(img, cmap='gray')
    color = 'green' if preds[i].item() == test_labels[i].item() else 'red'
    ax.set_title(f'True: {test_labels[i].item()}, Pred: {preds[i].item()}', color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 说明

1. **模型架构**: 使用 Transformer 编码器将 MNIST 图像视为 784 个 token 的序列
2. **训练**: 使用 AdamW 优化器 + Cosine Annealing 学习率调度
3. **完全自包含**: 所有代码内嵌，无需外部模块导入